# Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
import joblib

# Loading of processed data

In [2]:
df = pd.read_parquet('../data/processed/backrooms_processed.parquet')
# df = pd.read_csv('../data/processed/streamlit_test.csv')

X = df.drop(['survived_24h', 'escape_probability', 'survival_time_hours'], axis=1)
# X = df.drop(['survived_24h'], axis=1) # Удалить потом
y = df['survived_24h']

# Train, Val Splitting

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y) # train

In [4]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy='majority', random_state=42)
X_train, y_train = rus.fit_resample(X_train, y_train)

/Users/zimbazo/Documents/CodeProjects/BackroomsML/venv/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/zimbazo/Documents/CodeProjects/BackroomsML/venv/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


In [5]:
X_test.to_parquet('../data/processed/X_test.parquet', index=False)
y_test.to_frame().to_parquet('../data/processed/y_test.parquet', index=False)

# Baseline Model


In [6]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print('Baseline score: ', model.score(X_test, y_test))

Baseline score:  0.6084


/Users/zimbazo/Documents/CodeProjects/BackroomsML/venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Advanced Models + Tuning

## Model dict

In [7]:
# models = {
#     'LogisticRegression': LogisticRegression(),
#     'RandomForestClassifier': RandomForestClassifier(),
#     'GradientBoostingClassifier': GradientBoostingClassifier(),
#     'XGBoost': xgb.XGBClassifier(

#     ),
#     'LightGBM': lgb.LGBMClassifier(
        
#     )
# }

In [8]:
rf = xgb.XGBClassifier(random_state=42)

param_grid = {
    'n_estimators': [1000],
    'max_depth': [1, 2, 3, 5],
    'learning_rate': [0.01],
}

grid = GridSearchCV(rf, param_grid, cv=2, scoring='roc_auc')
grid.fit(X_train, y_train)

best_model = grid.best_estimator_

In [9]:
print(f'score: {best_model.score(X_test, y_test)}')
print(f'params: {best_model.get_params()}')

score: 0.6396
params: {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': None, 'feature_types': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.01, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 3, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 1000, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': None, 'validate_parameters': None, 'verbosity': None}


# Model saving

In [10]:
joblib.dump(best_model, '../models/best_model_v1.pkl')
print('best model has been saved to /models')

best model has been saved to /models
